# BME Lab 114 — Morphology-Elasticity Relationship of Trabecular Bone

**Author:** Simone Poncioni, MSB  
**Date:** Spring Semester 2026

---

## Notebook 3: Simulation

In the previous notebook, we generated a voxel-based FE mesh from the segmented bone image and exported it as an Abaqus-compatible `.inp` file. We are now ready to define the **physics of the problem** and run the simulation.

This notebook covers:

1. Inspecting the simulation template (boundary conditions & material properties)
2. Running the CalculiX FE solver

### 0. Imports

In [7]:
import subprocess
from pathlib import Path

### 1. Inspect the Simulation Template

Before running the solver, it is important to understand what physics we are imposing. The `.inp` template file defines:

- **Material:** cortical bone tissue modelled as a linear elastic isotropic material with Young's modulus $E = 18{,}000$ MPa and Poisson's ratio $\nu = 0.3$
- **Boundary conditions:** all degrees of freedom fixed at the bottom face (`NODES_Z0`); a compressive displacement of $-0.01$ mm imposed at the top face (`NODES_Z1`)
- **Output requests:** nodal displacements `U`, reaction forces `RF` at the bottom, and element stresses `S` and strains `E`

This setup simulates a **uniaxial compression test**, analogous to what is performed experimentally on bone cores.

> **Task:** Identify which lines in the template control the applied displacement and the material stiffness. What would you change to simulate a stiffer or softer tissue?

In [8]:
!cat "/home/bmelab/bmelabs/2026/testing/MSB_BMELab/114-MorphologyElasticityTrabecularBone/muFE/01_CODE/utils/template_sim.inp"

** ----------------------------------------------------
** Material definition:
*MATERIAL, NAME=BONE
*ELASTIC
18000, 0.3, 0
*SOLID SECTION, ELSET=SET1, MATERIAL=BONE
** ---------------------------------------------------
** Analysis definition. The Step module defines the analysis steps and associated output requests.
** More info at:
** https://abaqus-docs.mit.edu/2017/English/SIMACAECAERefMap/simacae-m-Sim-sb.htm#simacae-m-Sim-sb
** ---------------------------------------------------
*STEP
*STATIC
** All displacements fixed at bottom:
*BOUNDARY
NODES_Z0, 1, 3, 0.0
** Vertical displacement imposed at top:
*BOUNDARY
NODES_Z1, 3, 3, -0.01
** ---------------------------------------------------
** Output request:
*NODE FILE, OUTPUT=2D
U
*NODE PRINT, TOTALS=ONLY, NSET=NODES_Z0
RF
*EL FILE, OUTPUT=2D
S, E
*END STEP

### 2. Run the CalculiX Solver

We submit the `.inp` file to **CalculiX** (`ccx`), an open-source FE solver compatible with the Abaqus input format. The solver computes nodal displacements and element stresses across the entire mesh.

> **Task:** Once the solver finishes, note the number of nodes, elements, and degrees of freedom. How does this relate to the downsampling factor `DS` you chose?

In [9]:
inp_path = Path("../../00_DATA/group01/A1/derived/C0004351_segmented")

result = subprocess.run(
    f"export OMP_NUM_THREADS=12 && ccx '{inp_path}'",
    shell=True,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)


************************************************************

CalculiX Version 2.15, Copyright(C) 1998-2018 Guido Dhondt
CalculiX comes with ABSOLUTELY NO WARRANTY. This is free
software, and you are welcome to redistribute it under
certain conditions, see gpl.htm

************************************************************

You are using an executable made on Tue Dec 18 07:40:20 UTC 2018

  The numbers below are estimated upper bounds

  number of:

   nodes:       443322
   elements:       130214
   one-dimensional elements:            0
   two-dimensional elements:            0
   integration points per element:            8
   degrees of freedom per node:            3
   layers per element:            1

   distributed facial loads:            0
   distributed volumetric loads:            0
   concentrated loads:            0
   single point constraints:        10109
   multiple point constraints:            1
   terms in all multiple point constraints:            1
   tie constr